# Строим LLM


Этот ноутбук начинается с самого первого слоя будущей языковой модели: подготовки текста. До нейросети текст нельзя передать напрямую, потому что модель работает с числами. Поэтому путь выглядит так:

```text
исходный текст -> токены -> id токенов -> обучающие примеры
```

Здесь важно не просто получить числа, а сохранить связь между числом и фрагментом текста. Тогда модель сможет учиться предсказывать следующий токен, а после предсказания мы сможем перевести id обратно в читаемый текст.


## Токенизация текста


Токенизация - это разбиение текста на маленькие части, с которыми дальше будет работать модель. Токеном может быть слово, знак пунктуации, пробел, часть слова или специальный служебный маркер.

Для LLM токенизация особенно важна: модель не видит "слова" как человек. Она видит последовательность id, и качество этого разбиения влияет на размер словаря, длину последовательностей и то, насколько удобно модели учить закономерности языка.


Сначала загружается исходный учебный текст из файла `the-verdict.txt`. Переменная `raw_text` хранит весь текст одной строкой.

Две проверки после чтения помогают быстро понять масштаб данных:

- `len(raw_text)` показывает количество символов;
- `raw_text[:99]` выводит начало текста, чтобы убедиться, что файл прочитан правильно.

На этом этапе текст еще не подготовлен для модели: это просто обычная строка Python.


In [1]:
# Загружаем учебный текст и смотрим его размер и начало.
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Tota; number of character:", len(raw_text))
print(raw_text[:99])

Tota; number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


Теперь текст разбивается на грубые токены с помощью регулярного выражения. В `re.split(...)` скобки вокруг шаблона важны: они говорят Python сохранить разделители в результате.

Поэтому в список попадают не только слова, но и знаки пунктуации, пробелы и переносы строк. Для языковой модели это полезно, потому что пунктуация и пробельные символы тоже несут информацию о структуре текста.

Этот способ токенизации простой и учебный. Он хорошо показывает идею, но позже будет заменен на более практичный BPE-токенизатор.


In [2]:
# Делим текст на слова, пробелы и знаки пунктуации.
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
print(len(preprocessed))
print(preprocessed[:99])

9235
['I', ' ', 'HAD', ' ', 'always', ' ', 'thought', ' ', 'Jack', ' ', 'Gisburn', ' ', 'rather', ' ', 'a', ' ', 'cheap', ' ', 'genius', '--', 'though', ' ', 'a', ' ', 'good', ' ', 'fellow', ' ', 'enough', '--', 'so', ' ', 'it', ' ', 'was', ' ', 'no', ' ', 'great', ' ', 'surprise', ' ', 'to', ' ', 'me', ' ', 'to', ' ', 'hear', ' ', 'that', ',', '', ' ', 'in', ' ', 'the', ' ', 'height', ' ', 'of', ' ', 'his', ' ', 'glory', ',', '', ' ', 'he', ' ', 'had', ' ', 'dropped', ' ', 'his', ' ', 'painting', ',', '', ' ', 'married', ' ', 'a', ' ', 'rich', ' ', 'widow', ',', '', ' ', 'and', ' ', 'established', ' ', 'himself', ' ', 'in', ' ', 'a']


## Преобразование токенов в идентификаторы токенов


После токенизации нужно перейти от текстовых фрагментов к числам. Нейросеть не умеет напрямую принимать строку `"the"` или знак `","`, но может принимать id вроде `132` или `7`.

Словарь токенов решает эту задачу: каждому уникальному токену ставится в соответствие целое число. Эта таблица соответствий должна быть стабильной: один и тот же токен всегда должен получать один и тот же id.


Здесь из списка токенов собираются уникальные значения. `set(preprocessed)` удаляет повторы, а `sorted(...)` делает порядок стабильным и воспроизводимым.

Стабильный порядок важен для учебного примера: если словарь строится каждый раз одинаково, то id токенов тоже не будут случайно меняться между запусками.


In [3]:
# Собираем отсортированный список уникальных токенов.
all_words = sorted(set(preprocessed))
print(all_words[:99])

['', '\n', ' ', '!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be', 'Begin', 'Burlington', 'But', 'By', 'Carlo', 'Chicago', 'Claude', 'Come', 'Croft', 'Destroyed', 'Devonshire', 'Don', 'Dubarry', 'Emperors', 'Florence', 'For', 'Gallery', 'Gideon', 'Gisburn', 'Gisburns', 'Grafton', 'Greek', 'Grindle', 'Grindles', 'HAD', 'Had', 'Hang', 'Has', 'He', 'Her', 'Hermia', 'His', 'How', 'I', 'If', 'In', 'It', 'Jack', 'Jove', 'Just', 'Lord', 'Made', 'Miss', 'Money', 'Monte', 'Moon-dancers', 'Mr', 'Mrs', 'My', 'Never', 'No', 'Now', 'Nutley', 'Of', 'Oh', 'On', 'Once', 'Only', 'Or', 'Perhaps', 'Poor', 'Professional', 'Renaissance', 'Rickham', 'Riviera', 'Rome', 'Russian', 'Sevres', 'She', 'Stroud', 'Strouds', 'Suddenly', 'That', 'The', 'Then', 'There']


Теперь строится словарь `vocab`, где ключ - текстовый токен, а значение - его числовой id.

```text
token -> integer id
```

Функция `enumerate(all_words)` выдает пары вида `(номер, токен)`. В генераторе словаря они разворачиваются в соответствие `token: integer`. Первые элементы выводятся просто для проверки, что словарь действительно создан.


In [4]:
# Строим словарь token -> id и просматриваем первые элементы.
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('', 0)
('\n', 1)
(' ', 2)
('!', 3)
('"', 4)
("'", 5)
('(', 6)
(')', 7)
(',', 8)
('--', 9)
('.', 10)
(':', 11)
(';', 12)
('?', 13)
('A', 14)
('Ah', 15)
('Among', 16)
('And', 17)
('Are', 18)
('Arrt', 19)
('As', 20)
('At', 21)
('Be', 22)
('Begin', 23)
('Burlington', 24)
('But', 25)
('By', 26)
('Carlo', 27)
('Chicago', 28)
('Claude', 29)
('Come', 30)
('Croft', 31)
('Destroyed', 32)
('Devonshire', 33)
('Don', 34)
('Dubarry', 35)
('Emperors', 36)
('Florence', 37)
('For', 38)
('Gallery', 39)
('Gideon', 40)
('Gisburn', 41)
('Gisburns', 42)
('Grafton', 43)
('Greek', 44)
('Grindle', 45)
('Grindles', 46)
('HAD', 47)
('Had', 48)
('Hang', 49)
('Has', 50)


`SimpleTokenizerV1` упаковывает две операции в один класс:

- `encode(text)` превращает текст в список id;
- `decode(ids)` превращает список id обратно в текст.

Внутри хранятся два словаря. `str_to_int` нужен для кодирования, а `int_to_str` - для обратного преобразования. Это минимальная версия токенизатора: она работает только с токенами, которые уже есть в словаре.


In [5]:
# Описываем простой токенизатор на основе готового словаря.
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        # Создает обратный словарь, проецирующий идентификаторы в токены
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        """Текст в идентификаторы токенов."""
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """Токены в текст."""
        text = " ".join([self.int_to_str[i] for i in ids])
        # Удаляет пробелы перед определенным знаком пунктуации
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


Проверяем `encode` на коротком фрагменте текста. Токенизатор сначала разбивает строку тем же регулярным выражением, затем заменяет каждый токен его id из словаря.

Результат `ids` - это уже формат, с которым может работать модель: последовательность целых чисел. Пока это еще не тензор PyTorch, но смысловой переход уже произошел: текст стал числовой последовательностью.


In [6]:
# Проверяем кодирование фразы в числовые идентификаторы.
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
    Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[4, 59, 5, 853, 991, 605, 536, 749, 8, 1129, 599, 8, 4, 70, 10, 41, 854, 1111, 757, 796, 10]


Декодирование проверяет обратимость токенизации. Если `decode(encode(text))` возвращает почти тот же текст, значит связь между токенами и id работает правильно.

Небольшая чистка пробелов перед пунктуацией нужна потому, что при `join` между всеми токенами сначала вставляются пробелы, а потом лишние пробелы перед `,`, `.`, `?` и другими знаками удаляются регулярным выражением.


In [7]:
# Декодируем идентификаторы обратно в текст.
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


У первой версии токенизатора есть важное ограничение: если встретится слово, которого нет в словаре, кодирование упадет с ошибкой. Для реального текста это почти неизбежно.

Например, если в обучающем корпусе не было слова `Hello`, то `SimpleTokenizerV1` не знает, какой id ему поставить. Поэтому дальше добавляется специальный токен для неизвестных слов.


In [8]:
# Показываем пример слова, которого нет в словаре.
# Слово отсуствующее в словаре
# text = "Hello, do you like tea?"
# print(tokenizer.encode(text))

## Добавление контекстных токенов


Специальные токены не обязательно соответствуют обычным словам из текста. Они нужны, чтобы явно передавать модели служебную информацию.

В этом ноутбуке используются два таких токена:

- `<|endoftext|>` - маркер границы между двумя независимыми текстами;
- `<[unk]>` - маркер неизвестного токена, которого нет в словаре.

Без таких маркеров модели сложнее отличить настоящий текст от технических ситуаций вроде "документ закончился" или "слово не найдено".


Здесь словарь расширяется специальными токенами. Они добавляются в конец списка `all_tokens`, а затем словарь `vocab` строится заново.

Количество элементов печатается до и после пересборки, чтобы увидеть, что словарь действительно стал больше. Для модели это означает: теперь у служебных ситуаций тоже есть свои числовые id.


In [9]:
# Добавляем специальные токены конца текста и неизвестного слова.
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<[unk]>"])
print(len(vocab.items()))
vocab = {token:integer for integer, token in enumerate(all_tokens)}
print(len(vocab.items()))

1133
1135


Проверяем последние элементы словаря. Так удобно убедиться, что `<|endoftext|>` и `<[unk]>` попали в `vocab` и получили id.

В больших проектах такие проверки помогают ловить неприятные ошибки: если специальный токен не попал в словарь, токенизатор не сможет корректно обработать конец текста или неизвестное слово.


In [10]:
# Проверяем, что специальные токены попали в конец словаря.
for element in list(vocab.items())[-5:]:
    print(element)

('younger', 1130)
('your', 1131)
('yourself', 1132)
('<|endoftext|>', 1133)
('<[unk]>', 1134)


`SimpleTokenizerV2` решает две проблемы первой версии.

Во-первых, он не падает на неизвестных токенах: если токена нет в словаре, используется id для `<[unk]>`. Во-вторых, он аккуратнее обращается со специальными токенами, чтобы строка `<|endoftext|>` не была случайно разбита регулярным выражением на отдельные символы.

Это все еще учебный токенизатор, но он уже показывает важный принцип: tokenizer должен быть устойчивым к тексту, который немного отличается от обучающего корпуса.


In [11]:
# Улучшаем токенизатор: добавляем обработку unknown и special tokens.
import re

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        # Более аккуратное разбиение + защита special tokens
        # Сначала заменяем special tokens на временный маркер, чтобы их не разбило
        special_tokens = ["<|endoftext|>", "<[unk]>"]

        for token in special_tokens:
            text = text.replace(token, f" {token} ")  # окружаем пробелами для надёжного выделения

        # Разбиваем
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        # Заменяем неизвестные токены
        ids = []
        for item in preprocessed:
            if item in self.str_to_int:
                ids.append(self.str_to_int[item])
            else:
                ids.append(self.str_to_int['<[unk]>'])

        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        text = re.sub(r'\s+([<|])', r'\1', text)   # чистим пробелы перед special tokens
        return text

Теперь два отдельных текста склеиваются через `<|endoftext|>`. Такой маркер говорит модели: здесь закончился один фрагмент и начался другой.

Без явной границы модель могла бы воспринимать конец первого текста и начало второго как обычное продолжение одной фразы. Для языкового моделирования это может быть шумом, особенно если в корпусе много независимых документов.


In [12]:
# Склеиваем два текста специальным маркером конца текста.
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = "<|endoftext|>".join((text1, text2))
print(text)

Hello, do you like tea?<|endoftext|>In the sunlit terraces of the palace.


Здесь проверяется полный цикл `encode -> decode` для второй версии токенизатора. Это хороший тест на здравый смысл:

```text
текст -> id -> текст
```

Если после декодирования special token сохранился, а неизвестные слова не ломают программу, значит токенизатор стал заметно надежнее.


In [13]:
# Проверяем кодирование и декодирование токенизатором V2.
tokenizer = SimpleTokenizerV2(vocab)
encoded = tokenizer.encode(text)
print(encoded)
decoded = tokenizer.decode(encoded)
print(decoded)

[1134, 8, 358, 1129, 631, 978, 13, 1133, 58, 991, 959, 987, 725, 991, 1134, 10]
<[unk]>, do you like tea?<|endoftext|> In the sunlit terraces of the<[unk]>.


## Кодирование пар байтов


Кодирование пар байтов, или BPE, - более практичный подход к токенизации. Вместо того чтобы хранить только целые слова, BPE умеет разбивать редкие слова на частые подчасти.

Например, незнакомое слово можно представить не как `<[unk]>`, а как несколько известных фрагментов. Поэтому словарь остается не слишком большим, но tokenizer все равно может обработать почти любой текст.

GPT-2 использует именно такой тип токенизации, поэтому дальше берется готовая реализация из библиотеки `tiktoken`.


Подключаем готовый GPT-2 tokenizer. В отличие от учебных `SimpleTokenizerV1/V2`, он уже содержит заранее обученный словарь и правила BPE-разбиения.

Это ближе к реальной практике: для современных LLM обычно не пишут tokenizer с нуля для каждого эксперимента, а используют проверенную реализацию, совместимую с выбранной моделью.


In [14]:
# Подключаем готовый GPT-2 токенизатор из tiktoken.
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

Перед кодированием еще раз выводится примерный текст. Это помогает держать рядом исходную строку и результат токенизации: сначала видим человекочитаемый вариант, затем список id.


In [15]:
# Выводим пример текста для токенизации GPT-2.
text

'Hello, do you like tea?<|endoftext|>In the sunlit terraces of the palace.'

GPT-2 tokenizer превращает текст в список числовых id. Параметр `allowed_special={"<|endoftext|>"}` явно разрешает использовать этот специальный токен как единый токен.

Если special token не разрешить, tokenizer может считать его обычной строкой или выбросить ошибку, потому что такие маркеры требуют осознанной обработки.


In [16]:
# Кодируем текст GPT-2 токенизатором, разрешая special token.
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
integers

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 50256,
 818,
 262,
 4252,
 18250,
 8812,
 2114,
 286,
 262,
 20562,
 13]

Декодирование BPE-токенов возвращает строку. Это нужно не только для проверки: когда LLM генерирует новые id, именно tokenizer переводит их обратно в текст для пользователя.

Можно думать о tokenizer как о переводчике между двумя мирами:

```text
человеческий текст <-> числовая последовательность модели
```


In [17]:
# Восстанавливаем текст из GPT-2 токенов.
strings = tokenizer.decode(integers)
strings

'Hello, do you like tea?<|endoftext|>In the sunlit terraces of the palace.'

## Выборка данных с помощью контекстного окна


Теперь из длинной последовательности токенов нужно сделать обучающие примеры. Языковая модель обучается предсказывать следующий токен по предыдущим.

Контекстное окно - это ограниченное количество токенов, которое модель видит перед предсказанием. Если `context_size = 4`, значит модель получает до четырех предыдущих токенов и должна угадать следующий.


Здесь весь рассказ кодируется GPT-2 токенизатором. Переменная `enc_text` - это длинный список id, где каждый id соответствует одному BPE-токену.

`len(enc_text)` показывает длину текста уже не в символах, а в токенах. Для LLM это более важная величина, потому что именно токены являются единицами входа и предсказания.


In [18]:
# Токенизируе весь рассказ с токенизатором BPE
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
len(enc_text)

5145

Для демонстрации берутся первые 50 токенов. Полный текст может быть длинным, а маленький фрагмент удобнее использовать, чтобы руками увидеть принцип построения обучающих пар.

`enc_sample` - это не новый tokenizer и не новый текст, а просто небольшой срез уже закодированной последовательности.


In [19]:
# Смотрим первые 50 значений токенизирванного текста
enc_sample = enc_text[:50]
enc_sample

[40,
 367,
 2885,
 1464,
 1807,
 3619,
 402,
 271,
 10899,
 2138,
 257,
 7026,
 15632,
 438,
 2016,
 257,
 922,
 5891,
 1576,
 438,
 568,
 340,
 373,
 645,
 1049,
 5975,
 284,
 502,
 284,
 3285,
 326,
 11,
 287,
 262,
 6001,
 286,
 465,
 13476,
 11,
 339,
 550,
 5710,
 465,
 12036,
 11,
 6405,
 257,
 5527,
 27075,
 11]

Один из самых простых способов создания пар "входные данные - цель" для предсказания следующего слова - это создать две переменные х и у, где х содержит входные токены, а у - цели, которые являются входными токенами сдвинутыми на 1: 


Идея сдвига на один токен лежит в основе обучения языковой модели:

```text
x: [t0, t1, t2, t3]
y: [t1, t2, t3, t4]
```

Для каждой позиции модель смотрит на текущий контекст и учится предсказывать следующий токен. Поэтому `y` - это та же последовательность, что и `x`, но сдвинутая на один шаг вперед.

Такой формат позволяет из одного длинного текста получить много маленьких учебных примеров без ручной разметки.


Здесь явно создаются первые `x` и `y` для окна размера 4. `x` содержит четыре входных токена, а `y` содержит четыре целевых токена.

Важно, что цель для первого входного токена - второй токен, цель для второго - третий, и так далее. Модель учится не переводить весь текст сразу, а делать много локальных предсказаний "какой токен следующий".


In [20]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: {x}")
print(f"y: {y}")

x: [40, 367, 2885, 1464]
y: [367, 2885, 1464, 1807]


Этот цикл показывает идею растущего контекста. Сначала модель могла бы видеть один токен и предсказывать второй. Потом видеть два токена и предсказывать третий. Потом три - и так далее.

```text
[t0]             -> t1
[t0, t1]         -> t2
[t0, t1, t2]     -> t3
[t0, t1, t2, t3] -> t4
```

Так легче интуитивно понять задачу next-token prediction: следующий токен должен быть предсказан из всего доступного слева контекста.


In [21]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    # Слева от стрелки - входные данные, а справа - целевые
    print(context, " ----> ", desired)

[40]  ---->  367
[40, 367]  ---->  2885
[40, 367, 2885]  ---->  1464
[40, 367, 2885, 1464]  ---->  1807


Та же самая схема выводится уже в виде текста, а не числовых id. Это полезно для обучения: числа показывают реальный формат данных для модели, а декодированный текст помогает человеку увидеть смысл примеров.

Если слева стоит фрагмент фразы, справа находится следующий токен, который модель должна научиться предсказывать.


In [22]:
# Преобразование идентификаторов в текст
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    # Слева от стрелки - входные данные, а справа - целевые
    print(tokenizer.decode(context), " ----> ", tokenizer.decode([desired]))

I  ---->   H
I H  ---->  AD
I HAD  ---->   always
I HAD always  ---->   thought


### Что важно запомнить

В этой части ноутбука строится полный путь от текста к учебным примерам:

- исходный текст читается как строка;
- строка разбивается на токены;
- каждый уникальный токен получает числовой id;
- tokenizer умеет кодировать текст в id и декодировать id обратно;
- special tokens помогают обозначать конец текста и неизвестные токены;
- BPE tokenizer GPT-2 работает с частями слов и лучше подходит для реальных текстов;
- контекстное окно превращает длинную последовательность id в пары `x -> y` для предсказания следующего токена.

Это фундамент LLM: до архитектуры Transformer модель сначала должна получить данные в правильном числовом формате.
